In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("trading_sentiment_analysis").getOrCreate()

In [0]:
filings_df = (
    spark.read.table("workspace.sec_filings.stg_clean_10Ks")
    )

display(filings_df)

In [0]:
from pyspark.sql import functions as F
lm_df = (
    spark.read
    .option("header", "True")
    .csv("/Volumes/workspace/sec_filings/filings/Loughran-McDonald_MasterDictionary_1993-2024.csv"))

lm_df = (
    lm_df
    .withColumn("positive_bool", F.when(F.col("Positive") != 0, 1).otherwise(0))
    .withColumn("negative_bool", F.when(F.col("Negative") != 0, 1).otherwise(0))
    .withColumn("uncertainty_bool", F.when(F.col("Uncertainty") != 0, 1).otherwise(0))
    .withColumn("litigious_bool", F.when(F.col("Litigious") != 0, 1).otherwise(0))
    .withColumn("constraining_bool", F.when(F.col("Constraining") != 0, 1).otherwise(0))
    )
display(lm_df)

In [0]:

from pyspark.sql import functions as F
lm_df = (
    lm_df
    .select(F.lower("Word").alias("word"),
        F.col("positive_bool").cast("boolean").alias("is_positive"),
        F.col("negative_bool").cast("boolean").alias("is_negative"),
        F.col("uncertainty_bool").cast("boolean").alias("is_uncertainty"),
        F.col("litigious_bool").cast("boolean").alias("is_litigious"),
        F.col("constraining_bool").cast("boolean").alias("is_constraining")
        )
    )

display(lm_df)
display(lm_df.count())
lm_df.dropDuplicates(["word"]).write.format("delta").mode("overwrite").saveAsTable("sec_filings.lm_lexicon")

In [0]:
sections = (
    filings_df
    .select(
        "cik", "company_name", "tickers",
        "filing_date", "accessionNumber",
        "market_cap", "sector",
        F.col("bus_text").alias("BUSINESS"),
        F.col("risks_text").alias("RISKS"),
        F.col("mda_text").alias("MDA")
    )
    .selectExpr(
        "cik", "company_name", "tickers",
        "filing_date", "accessionNumber",
        "market_cap", "sector",
        "stack(3, 'BUSINESS', BUSINESS, 'RISKS', RISKS, 'MDA', MDA) as (section_name, section_text)"
    )
    .where("section_text is not null and length(trim(section_text)) > 0")
)

cleaned = (
    sections
    .withColumn("section_text_lower",
                F.lower(F.col("section_text")))
    .withColumn(
        "tokens",
        F.split(
            F.regexp_replace("section_text_lower", r"[^a-z0-9 ]", " "),
            r"\s+"
        )
    )
    .withColumn("token", F.explode("tokens"))
    .where("token != ''")
)

display(cleaned)
display(cleaned.count())

In [0]:
# join with lm lexicon and aggregate count

lm = spark.read.table("sec_filings.lm_lexicon")

scored_tokens = (
    cleaned
    .join(lm, cleaned.token == lm.word, "left")
)

agg = (
    scored_tokens
    .groupBy(
        "cik","company_name","tickers",
        "filing_date","accessionNumber",
        "market_cap","sector",
        "section_name"
    )
    .agg(
        F.count("*").alias("total_tokens"),
        F.sum(F.when(F.col("is_positive"), 1).otherwise(0)).alias("pos_count"),
        F.sum(F.when(F.col("is_negative"), 1).otherwise(0)).alias("neg_count"),
        F.sum(F.when(F.col("is_uncertainty"), 1).otherwise(0)).alias("uncertainty_count"),
        F.sum(F.when(F.col("is_litigious"), 1).otherwise(0)).alias("litigious_count"),
        F.sum(F.when(F.col("is_constraining"), 1).otherwise(0)).alias("constraining_count")
    )
    .orderBy(F.desc("market_cap"), F.desc("filing_date"))
)
display(agg)
display(agg.count())

In [0]:
# Map to the three buckets: Optimism, Caution, Concern
from pyspark.sql import functions as F
agg_scored = (
    agg
    .withColumn("optimism_raw", F.col("pos_count"))
    .withColumn("caution_raw", F.col("uncertainty_count") + F.col("constraining_count"))
    .withColumn("concern_raw", F.col("neg_count") + F.col("litigious_count"))
    .withColumn("token_k", F.col("total_tokens") / F.lit(1000.0))
    .withColumn("optimism_per_1k", F.col("optimism_raw") / F.col("token_k"))
    .withColumn("caution_per_1k", F.col("caution_raw") / F.col("token_k"))
    .withColumn("concern_per_1k", F.col("concern_raw") / F.col("token_k"))
    .withColumn("total_signal",
                F.col("optimism_raw") + F.col("caution_raw") + F.col("concern_raw"))
    .withColumn("optimism_share",
                F.col("optimism_raw") / F.col("total_signal"))
    .withColumn("caution_share",
                F.col("caution_raw") / F.col("total_signal"))
    .withColumn("concern_share",
                F.col("concern_raw") / F.col("total_signal"))
    .orderBy(F.desc("market_cap"), F.desc("filing_date"))
)
display(agg_scored)

# (agg_scored
#  .write
#  .mode("overwrite")
#  .saveAsTable("sec_filings.fact_lm_sentiment"))


In [0]:
lm = spark.read.table("workspace.sec_filings.lm_lexicon")


display(lm.count())

In [0]:
spark.sql("SHOW TABLES IN workspace.sec_filings").display()

In [0]:
spark.sql("DROP TABLE IF EXISTS sec_filings.lm_lexicon")